# Simple workflow
Learn following using OpenAI framework
1. Agent workflow 
2. Tools 
3. Handoffs
4. Guardrails



In [16]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, input_guardrail, GuardrailFunctionOutput
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import asyncio
import os
from pydantic import BaseModel

load_dotenv(override=True)

True

# Step 1: Agent Workflow

In [5]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [6]:
sales_agent1 = Agent(
    name="Professional Sales Agent",
    instructions=instructions1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="Engaging Sales Agent",
    instructions=instructions2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Busy Sales Agent",
    instructions=instructions3,
    model="gpt-4o-mini"
)

In [4]:
# Test the connectivity to openai by running simple check

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Simplify Your SOC 2 Compliance with ComplAI

Dear [Recipient’s Name],

I hope this message finds you well. My name is [Your Name], and I'm with ComplAI, an AI-powered SaaS solution designed specifically to streamline SOC 2 compliance processes and enhance audit preparedness.

Navigating the complexities of SOC 2 compliance can be a daunting task, often diverting valuable resources away from your core business activities. ComplAI offers an intelligent platform that automates compliance workflows, provides real-time insights, and simplifies documentation, allowing you to focus on what you do best.

Here are a few ways our solution can benefit your organization:

- **Automated Compliance Management**: Reduce the manual workload with automated tracking and reporting.
- **Real-Time Insights**: Gain immediate visibility into your compliance posture.
- **Audit-Ready Documentation**: Elevate your readiness for audits with organized, easily accessible records.

I would love to schedule

### Run all 3 agents in parallel

In [5]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Streamline Your SOC 2 Compliance with ComplAI

Dear [Recipient's Name],

I hope this message finds you well.

As organizations face increasingly complex requirements for data security and compliance, ensuring SOC 2 certification can be both time-consuming and resource-intensive. At ComplAI, we specialize in simplifying this process through our cutting-edge SaaS tool, designed specifically to streamline SOC 2 compliance and audit preparation.

Our AI-powered platform not only provides real-time insights but also automates documentation and workflows, reducing the burden on your team while enhancing accuracy. By integrating our solution, you can expect to:

- Accelerate your SOC 2 audit timelines
- Minimize manual errors
- Maintain continuous compliance with evolving standards

I would love the opportunity to discuss how ComplAI can support your compliance efforts and help you save valuable time and resources.

Are you available for a quick call next week? 

Thank you for consid

In [7]:
# Agent to pick best email from the list

sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini"
)

### Generate 3 emails using 3 agents and pick the best one using another agent

In [8]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")

Best sales email:
Subject: Is Your SOC2 Compliance Feeling a Little... Stressed? 🚀

Hi [Recipient's Name],

I hope this email finds you with your coffee cup full and your stress levels low! I wanted to slide into your inbox like a perfectly executed audit checklist and sprinkle some magic on your compliance woes.

Introducing ComplAI – because keeping your SOC2 compliance on track shouldn’t feel like trying to assemble IKEA furniture without the instructions. 🤯 You know, confusing, frustrating, and an immediate candidate for a deep sigh?

Our AI-powered tool takes the guesswork out of compliance and audits. Think of us as your trusty sidekick, armed with a virtual toolkit that’s sharper than your coffee mug collection. We streamline the entire process, ensuring you're both complaint-ready and beautifully organized. You can focus on what you do best—growing your business—while we handle the compliance chaos!

Want to see how it works? Let’s set up a quick chat (don’t worry, no lengthy a

Now go and check out the trace:

https://platform.openai.com/traces

# 2. Tools
### Crete a tool using @function_tool decorator for the function

In [9]:
@function_tool
def send_email(body: str):
    """ This is a fake method and it just prints the email instead of sending it out.  In real world we could have send email routine using sendgrid or some other service/library """
    print("In send_email function" )
    return {"status": "success"}

In [20]:
# Lets look at the variable.. it should have created FunctionTool that can be sent to the model
send_email

FunctionTool(name='send_email', description='This is a fake method and it just prints the email instead of sending it out.  In real world we could have send email routine using sendgrid or some other service/library', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10ce6e980>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

# 3. Handoffs 
Calling agents give control to Handoff agents and Handoff agents take it forward from there and calling agent will not receive the reply

In [10]:
instructions ="You are an email sender. You receive the body of an email to be sent. \
You use the send_email tool to send the email "

email_tools = [send_email]

emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=email_tools,
    model="gpt-4o-mini",
    handoff_description="Send an email")


### Convert agents to tool.  Put all tools together into array


In [11]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent1.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent1.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3]

tools


[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10c8920c0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x10c891f80>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent3', description='Write 

### Sales Manager agent is a planning agent.  
It is given tools to generate 3 emails, instructions to select the best email and hand off agent to send the email

In [12]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Handoff for Sending: Pass ONLY the winning email draft to 'Email Manager' agent. The Email Manager will take care of sending email.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager", 
    instructions=instructions, 
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini")

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

In send_email function


### Now go and check out the trace to see all agents:

https://platform.openai.com/traces


# 4. Guardrails
Guardrails are added before sending information to model or after receiving the answer from the model.  

In [14]:
# Create a guardrail to check whether Name (PII) is being used in email

class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent = Agent(
    name="Name Check",
    instructions="check if the user is including someone's personal name in what they want you to do",
    output_type=NameCheckOutput,
    model="gpt-4o-mini"
)


In [17]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

### Use guardrail on input
Following will trip it right away as we are using name in the message.  It should trip the whole workflow with a message saying 'Guardrail InputGuardrail triggered tripwire'

In [19]:
careful_sales_manager = Agent(
    name = "Careful Sales Manager",
    instructions=instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_name]
)

message = "Send out a cold sales email addressed to Dear CEO Satish"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

### Check out the trace: 
https://platform.openai.com/traces

### Now try without passing the name so that it does NOT trip it

In [20]:

message = "Send out a cold sales email addressed to Dear CEO"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

In send_email function
In send_email function
In send_email function
